In [1]:
import pandas as pd

def main():
    print("Đang đọc file eprom.json...")
    df_eprom = pd.read_json('eprom.json')

    # 1. Lấy cột mã dài, đổi tên thành issued_promo_id làm Khóa chính (PK)
    df_issued = df_eprom[['promo_id']].rename(columns={'promo_id': 'issued_promo_id'})

    # 2. Tạo Khóa ngoại (FK) promo_id nối về bảng gốc
    # Hàm str.extract(r'(\d+)') sẽ chộp lấy cụm số đầu tiên nó thấy (vd: '0039' từ 'PROMO-0039-0016')
    # Sau đó astype(int) sẽ ép '0039' thành số nguyên 39 
    print("Đang xử lý Khóa ngoại thành số nguyên...")
    df_issued['promo_id'] = df_issued['issued_promo_id'].astype(str).str.extract(r'(\d+)')[0].astype(int)

    # 3. Xuất ra file CSV
    df_issued.to_csv('ISSUED_PROMOTIONS.csv', index=False, encoding='utf-8-sig')
    print("Đã tạo xong bảng ISSUED_PROMOTIONS.csv với FK là số!")

if __name__ == "__main__":
    main()

Đang đọc file eprom.json...
Đang xử lý Khóa ngoại thành số nguyên...
Đã tạo xong bảng ISSUED_PROMOTIONS.csv với FK là số!


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore') # Tắt các cảnh báo mặc định của Pandas

def advanced_clean_and_report(file_path, table_name, expected_cols, pk_col, fk_configs=None):
    print(f"\n{'='*70}")
    print(f"BẮT ĐẦU PHÂN TÍCH VÀ LÀM SẠCH BẢNG: {table_name}")
    print(f"{'='*70}")
    
    if not os.path.exists(file_path):
        print(f"[LỖI] Không tìm thấy file {file_path}")
        return None

    # Đọc dữ liệu
    df = pd.read_csv(file_path)
    
    # Chỉ lấy các cột có trong schema
    valid_cols = [col for col in expected_cols if col in df.columns]
    missing_cols = [col for col in expected_cols if col not in df.columns]
    if missing_cols:
        print(f"[*] Cảnh báo: File thô đang thiếu các cột sau: {missing_cols}")
        
    df = df[valid_cols].copy()

    # Xóa dòng rỗng hoàn toàn và duplicate nguyên dòng (giống hệt nhau 100%)
    initial_len = len(df)
    df = df.dropna(how='all').drop_duplicates()
    deleted_garbage = initial_len - len(df)

    # --- KIỂM TRA VÀ XÓA TRÙNG KHÓA CHÍNH (PK) ---
    pk_duplicates_removed = 0
    if pk_col in df.columns:
        # Tìm các dòng trùng Khóa chính
        pk_dup_mask = df.duplicated(subset=[pk_col], keep='first')
        pk_duplicates_removed = pk_dup_mask.sum()
        # Giữ lại dòng đầu tiên, đá văng các dòng trùng PK
        if pk_duplicates_removed > 0:
            df = df[~pk_dup_mask]

    # --- KHỞI TẠO BIẾN THỐNG KÊ ---
    stats = {
        "initial_rows": initial_len,
        "initial_cols": len(df.columns),
        "100_percent_missing": [],
        "partial_missing": [],
        "anomalies": [],
        "rounded_cols": [],
        "fk_reports": [],
        "changes": {
            "to_null_all": 0,
            "median": 0,
            "mean": 0,
            "mode": 0,
            "rounded": 0,
            "pk_generated": 0,
            "deleted_garbage": deleted_garbage,
            "pk_duplicates_removed": pk_duplicates_removed # Báo cáo số dòng đụng PK
        },
        "col_reports": []
    }

    # ==========================================
    # 1. XỬ LÝ RIÊNG CHO KHÓA CHÍNH (PK)
    # ==========================================
    if pk_col in df.columns:
        missing_pk_mask = df[pk_col].isnull()
        missing_pk_count = missing_pk_mask.sum()

        if missing_pk_count > 0:
            existing_pks = pd.to_numeric(df[pk_col], errors='coerce').dropna()
            max_id = int(existing_pks.max()) if not existing_pks.empty else 0
            
            new_ids = [max_id + i + 1 for i in range(missing_pk_count)]
            df.loc[missing_pk_mask, pk_col] = new_ids
            
            stats["changes"]["pk_generated"] += missing_pk_count
            stats["col_reports"].append((pk_col, missing_pk_count, "Tự tạo ID (Max + 1)", missing_pk_count, 0, f"Bắt đầu từ: {new_ids[0]}"))

    # ==========================================
    # 2. KIỂM TRA KHÓA NGOẠI (FK) - CHỈ BÁO CÁO, KHÔNG XÓA
    # ==========================================
    if fk_configs:
        for fk_conf in fk_configs:
            fk_col = fk_conf['fk_col']
            parent_file = fk_conf['parent_file']
            parent_pk = fk_conf['parent_pk']
            
            if fk_col in df.columns:
                if not os.path.exists(parent_file):
                    stats["fk_reports"].append(f"[LỖI FK] Thiếu file {parent_file} để đối chiếu cột {fk_col}.")
                    continue
                    
                df_parent = pd.read_csv(parent_file)
                if parent_pk not in df_parent.columns:
                    stats["fk_reports"].append(f"[LỖI FK] Cột {parent_pk} không có trong {parent_file}.")
                    continue
                    
                child_fks = df[fk_col].dropna().unique()
                parent_pks = df_parent[parent_pk].dropna().unique()
                orphan_fks = set(child_fks) - set(parent_pks)
                
                if orphan_fks:
                    orphan_records = df[df[fk_col].isin(orphan_fks)]
                    stats["fk_reports"].append({
                        "fk_col": fk_col,
                        "parent_file": parent_file,
                        "orphan_count": len(orphan_fks),
                        "orphan_values": list(orphan_fks)[:5],
                        "record_count": len(orphan_records)
                    })

    # ==========================================
    # 3. XỬ LÝ GIÁ TRỊ THIẾU CÁC CỘT CÒN LẠI
    # ==========================================
    for col in df.columns:
        if col == pk_col:
            continue
            
        missing_count = df[col].isnull().sum()
        if missing_count == 0:
            continue
            
        if missing_count == len(df):
            df[col] = np.nan 
            stats["100_percent_missing"].append(col)
            stats["changes"]["to_null_all"] += missing_count
            stats["col_reports"].append((col, missing_count, "Chuyển thành NaN (100% thiếu)", missing_count, 0, "Không xóa cột"))
            continue

        stats["partial_missing"].append(col)
        col_type = df[col].dtype
        replaced_val = None
        
        if pd.api.types.is_numeric_dtype(col_type):
            skewness = df[col].skew() if not pd.isna(df[col].skew()) else 0
            if abs(skewness) > 1: 
                replaced_val = df[col].median()
                method_used = f"Median (Skew={skewness:.2f})"
                stats["changes"]["median"] += missing_count
            else: 
                replaced_val = df[col].mean()
                method_used = f"Mean (Phân phối ổn)"
                stats["changes"]["mean"] += missing_count
        else:
            modes = df[col].mode()
            if not modes.empty:
                replaced_val = modes[0]
                method_used = "Mode"
                stats["changes"]["mode"] += missing_count
            else:
                method_used = "Giữ nguyên"

        if replaced_val is not None:
            df[col] = df[col].fillna(replaced_val)
            stats["col_reports"].append((col, missing_count, method_used, missing_count, 0, f"Thay bằng: {replaced_val}"))

    # ==========================================
    # 4. LÀM TRÒN SỐ & PHÁT HIỆN BẤT THƯỜNG
    # ==========================================
    for col in df.columns:
        if col == pk_col:
            continue
            
        # Làm tròn
        if pd.api.types.is_float_dtype(df[col]):
            rounded_series = df[col].round(2)
            changed_count = (df[col] != rounded_series).sum()
            if changed_count > 0:
                df[col] = rounded_series
                stats["rounded_cols"].append(col)
                stats["changes"]["rounded"] += changed_count
                stats["col_reports"].append((col, "-", "Làm tròn số", 0, changed_count, "Làm tròn 2 số thập phân"))

        # Tìm dòng bị trộn chữ vào số (anomalies)
        if df[col].dtype == object:
            temp_numeric = pd.to_numeric(df[col], errors='coerce')
            if temp_numeric.notna().sum() > 0 and temp_numeric.notna().sum() < len(df[col].dropna()):
                anomaly_mask = temp_numeric.isna() & df[col].notna()
                if anomaly_mask.sum() > 0:
                    stats["anomalies"].append({
                        "col": col,
                        "count": anomaly_mask.sum(),
                        "values": df.loc[anomaly_mask, col].unique()[:5]
                    })

    # ==========================================
    # OUTPUT BÁO CÁO 
    # ==========================================
    print("\n[1] TỔNG QUAN")
    print(f" - Số dòng ban đầu: {stats['initial_rows']}")
    print(f" - Số dòng rác (trống/trùng 100%) đã xóa: {stats['changes']['deleted_garbage']}")
    print(f" - Số dòng vi phạm duy nhất PK đã xóa: {stats['changes']['pk_duplicates_removed']}")
    
    print("\n[2] BÁO CÁO KHÓA NGOẠI (FK)")
    if not fk_configs:
        print(" - Không thiết lập kiểm tra Khóa ngoại.")
    elif not stats["fk_reports"]:
        print(" -> Hợp lệ 100%. Tất cả FK đều khớp với bảng cha.")
    else:
        for report in stats["fk_reports"]:
            if isinstance(report, str):
                print(f" {report}")
            else:
                print(f" [CẢNH BÁO] Cột '{report['fk_col']}' có {report['orphan_count']} giá trị không có trong {report['parent_file']}")
                print(f"    + Số dòng bị ảnh hưởng: {report['record_count']}")
                print(f"    + Vài giá trị mồ côi mẫu: {report['orphan_values']}")

    print("\n[3] GIÁ TRỊ THIẾU & LÀM TRÒN")
    print(f" - Cột missing 100%: {stats['100_percent_missing'] if stats['100_percent_missing'] else 'Không'}")
    print(f" - Cột missing 1 phần: {stats['partial_missing'] if stats['partial_missing'] else 'Không'}")
    print(f" - Số PK tự tạo (Max+1): {stats['changes']['pk_generated']}")
    
    if stats["anomalies"]:
        print("\n[4] DỮ LIỆU KHÔNG ĐỒNG NHẤT (Mixed Type)")
        for anomaly in stats["anomalies"]:
            print(f" - Cột '{anomaly['col']}': {anomaly['count']} dòng bất thường. Mẫu: {anomaly['values']}")

    print("\n[CHI TIẾT THAY ĐỔI TỪNG CỘT]")
    if stats["col_reports"]:
        report_df = pd.DataFrame(stats["col_reports"], columns=["Cột", "Missing gốc", "Phương pháp", "Đã thay", "Đã làm tròn", "Ghi chú"])
        print(report_df.to_string(index=False))
    else:
        print(" - Không có thay đổi nào.")

    output_filename = f"{table_name}_CLEANED.csv"
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n=> Đã lưu file thành công: {output_filename}")
    
    return df


if __name__ == "__main__":
    # Thiết lập config chuẩn cho ISSUED_PROMOTIONS
    ISSUED_CONFIG = {
        "file_raw": "ISSUED_PROMOTIONS.csv",
        "table_name": "ISSUED_PROMOTIONS",
        "columns": ["issued_promo_id", "promo_id"],
        "pk": "issued_promo_id",
        # Đối chiếu khóa ngoại promo_id về file PROMOTIONS(1) gốc
        "fks": [
            {
                "fk_col": "promo_id", 
                "parent_file": "PROMOTIONS (1).csv", 
                "parent_pk": "promo_id"
            }
        ]
    }

    advanced_clean_and_report(
        file_path=ISSUED_CONFIG["file_raw"],
        table_name=ISSUED_CONFIG["table_name"],
        expected_cols=ISSUED_CONFIG["columns"],
        pk_col=ISSUED_CONFIG["pk"],
        fk_configs=ISSUED_CONFIG["fks"]
    )


BẮT ĐẦU PHÂN TÍCH VÀ LÀM SẠCH BẢNG: ISSUED_PROMOTIONS

[1] TỔNG QUAN
 - Số dòng ban đầu: 1000
 - Số dòng rác (trống/trùng 100%) đã xóa: 0
 - Số dòng vi phạm duy nhất PK đã xóa: 0

[2] BÁO CÁO KHÓA NGOẠI (FK)
 [LỖI FK] Thiếu file PROMOTIONS(1).csv để đối chiếu cột promo_id.

[3] GIÁ TRỊ THIẾU & LÀM TRÒN
 - Cột missing 100%: Không
 - Cột missing 1 phần: Không
 - Số PK tự tạo (Max+1): 0

[CHI TIẾT THAY ĐỔI TỪNG CỘT]
 - Không có thay đổi nào.

=> Đã lưu file thành công: ISSUED_PROMOTIONS_CLEANED.csv
